# 05 — Neural Collaborative Filtering (NeuMF) — **Colab Edition**

This is a self-contained Colab-ready version. Steps:

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run cell **0a** below — installs deps, uploads your `~/.kaggle/access_token`, downloads H&M data
3. Run cell **0b** — writes `src/data.py` and `src/metrics.py` inline
4. Then run the rest of the notebook top-to-bottom

Total Colab time on T4: ~5–8 min for the full notebook.


In [ ]:
# Cell 0a — Colab setup: install deps, configure Kaggle, download data
import os, subprocess
from google.colab import files

print('Installing kaggle + implicit...')
subprocess.run(['pip', '-q', 'install', 'kaggle', 'implicit'], check=True)

print('Upload your kaggle access_token file (or kaggle.json):')
uploaded = files.upload()
fname = next(iter(uploaded))
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
if fname.endswith('.json'):
    target = os.path.expanduser('~/.kaggle/kaggle.json')
else:
    target = os.path.expanduser('~/.kaggle/access_token')
with open(target, 'wb') as f:
    f.write(uploaded[fname])
os.chmod(target, 0o600)

os.makedirs('/content/data', exist_ok=True)
for f in ['articles.csv', 'customers.csv', 'transactions_train.csv']:
    print(f'Downloading {f}...')
    subprocess.run(['kaggle','competitions','download','-c','h-and-m-personalized-fashion-recommendations','-f',f,'-p','/content/data'], check=True)
    z = f'/content/data/{f}.zip'
    if os.path.exists(z):
        subprocess.run(['unzip','-o',z,'-d','/content/data'], check=True)
        os.remove(z)
subprocess.run(['ls','-lh','/content/data'], check=True)


In [ ]:
# Cell 0b — write src/data.py and src/metrics.py inline
import os, textwrap
os.makedirs('/content/src', exist_ok=True)
DATA_PY = '''"""H&M dataset loaders and splitters.

The full transactions_train.csv (~31M rows) is too large for interactive
notebook work on a 16 GB laptop, so all loaders accept a `nrows` /
`sample_frac` argument. Use `nrows=None` only when you have enough RAM
or are running on Colab/Kaggle.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path(__file__).resolve().parents[1] / "data"


def load_articles(nrows: int | None = None) -> pd.DataFrame:
    """Load the article (item) catalogue."""
    df = pd.read_csv(DATA_DIR / "articles.csv", nrows=nrows, dtype={"article_id": str})
    return df


def load_customers(nrows: int | None = None) -> pd.DataFrame:
    """Load the customer profile table."""
    df = pd.read_csv(DATA_DIR / "customers.csv", nrows=nrows)
    return df


def load_transactions(
    nrows: int | None = None,
    sample_frac: float | None = None,
    last_months: int | None = None,
    parse_dates: bool = True,
    seed: int = 42,
    chunksize: int = 500_000,
) -> pd.DataFrame:
    """Load the implicit-feedback transactions.

    Three loading modes (use exactly one):
      - `nrows=N`: reads the first N rows (chronological head — earliest period).
        Avoid for modelling — chronologically biased.
      - `sample_frac=F`: reads the file in chunks and randomly keeps a fraction
        F of rows across the whole 31.8M-row span. Use for **EDA** (unbiased
        distribution statistics). Avoid for modelling — produces sparse user
        profiles (~2 train interactions per user at frac=0.03).
      - `last_months=M`: keeps only the last M calendar months of data. Use for
        **modelling** — matches H&M competition protocol and gives realistic
        user-profile density.

    All modes are memory-bounded by `chunksize`.
    """
    path = DATA_DIR / "transactions_train.csv"
    if sample_frac is not None and 0 < sample_frac < 1:
        rng = np.random.default_rng(seed)
        sampled_chunks = []
        for chunk in pd.read_csv(path, chunksize=chunksize, dtype={"article_id": str}):
            mask = rng.random(len(chunk)) < sample_frac
            if mask.any():
                sampled_chunks.append(chunk.loc[mask])
        df = pd.concat(sampled_chunks, ignore_index=True)
    elif last_months is not None and last_months > 0:
        # Two-pass: cheap scan to find max date, then keep rows in window
        max_date = pd.to_datetime(
            pd.read_csv(path, usecols=["t_dat"], dtype=str)["t_dat"].max()
        )
        cutoff = max_date - pd.DateOffset(months=last_months)
        sampled_chunks = []
        for chunk in pd.read_csv(path, chunksize=chunksize, dtype={"article_id": str}):
            chunk_dates = pd.to_datetime(chunk["t_dat"])
            mask = chunk_dates > cutoff
            if mask.any():
                sampled_chunks.append(chunk.loc[mask])
        df = pd.concat(sampled_chunks, ignore_index=True) if sampled_chunks else pd.DataFrame()
    else:
        df = pd.read_csv(path, nrows=nrows, dtype={"article_id": str})
    if parse_dates and len(df):
        df["t_dat"] = pd.to_datetime(df["t_dat"])
    return df


def time_based_split(
    transactions: pd.DataFrame,
    cutoff_days: int = 7,
    date_col: str = "t_dat",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split transactions into train / test by holding out the last N days.

    Time-based splits are more realistic than random splits for purchase data
    because they avoid leakage from the future into the past.
    """
    max_date = transactions[date_col].max()
    cutoff = max_date - pd.Timedelta(days=cutoff_days)
    train = transactions[transactions[date_col] <= cutoff].copy()
    test = transactions[transactions[date_col] > cutoff].copy()
    return train, test


def cold_user_ids(train: pd.DataFrame, test: pd.DataFrame, user_col: str = "customer_id") -> set[str]:
    """Users present in test but not in train (user-side cold-start)."""
    return set(test[user_col].unique()) - set(train[user_col].unique())


def cold_item_ids(train: pd.DataFrame, test: pd.DataFrame, item_col: str = "article_id") -> set[str]:
    """Items present in test but not in train (item-side cold-start)."""
    return set(test[item_col].unique()) - set(train[item_col].unique())


def build_user_item_matrix(
    transactions: pd.DataFrame,
    user_col: str = "customer_id",
    item_col: str = "article_id",
    weight: str | None = None,
):
    """Build a sparse CSR user-item interaction matrix.

    `weight` can be `None` (binary) or a column name to sum (e.g. quantity).
    Returns (matrix, user_index, item_index).
    """
    from scipy.sparse import csr_matrix

    users = pd.Index(transactions[user_col].unique(), name=user_col)
    items = pd.Index(transactions[item_col].unique(), name=item_col)
    user_pos = users.get_indexer(transactions[user_col])
    item_pos = items.get_indexer(transactions[item_col])
    values = transactions[weight].values if weight else np.ones(len(transactions), dtype=np.float32)
    matrix = csr_matrix(
        (values, (user_pos, item_pos)),
        shape=(len(users), len(items)),
        dtype=np.float32,
    )
    return matrix, users, items
'''
METRICS_PY = '''"""Top-N recommendation evaluation metrics.

All functions take ranked lists of predicted item IDs and a set of
ground-truth item IDs, evaluated at cutoff k.
"""

from __future__ import annotations

import numpy as np


def precision_at_k(predicted: list, actual: set, k: int) -> float:
    if k == 0:
        return 0.0
    top_k = predicted[:k]
    hits = sum(1 for p in top_k if p in actual)
    return hits / k


def recall_at_k(predicted: list, actual: set, k: int) -> float:
    if not actual:
        return 0.0
    top_k = predicted[:k]
    hits = sum(1 for p in top_k if p in actual)
    return hits / len(actual)


def hit_rate_at_k(predicted: list, actual: set, k: int) -> float:
    return 1.0 if any(p in actual for p in predicted[:k]) else 0.0


def average_precision_at_k(predicted: list, actual: set, k: int) -> float:
    if not actual:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)


def ndcg_at_k(predicted: list, actual: set, k: int) -> float:
    if not actual:
        return 0.0
    dcg = 0.0
    for i, p in enumerate(predicted[:k]):
        if p in actual:
            dcg += 1.0 / np.log2(i + 2)
    ideal_hits = min(len(actual), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate(recommendations: dict, ground_truth: dict, k: int = 10) -> dict:
    """Mean metrics across all evaluated users.

    `recommendations` and `ground_truth` are dicts: user_id -> list / set of item_ids.
    Users present in `ground_truth` but missing from `recommendations` are skipped
    (treat as cold-start for the algorithm being evaluated).
    """
    users = [u for u in ground_truth if u in recommendations]
    if not users:
        return {"users_evaluated": 0}
    p = np.mean([precision_at_k(recommendations[u], set(ground_truth[u]), k) for u in users])
    r = np.mean([recall_at_k(recommendations[u], set(ground_truth[u]), k) for u in users])
    h = np.mean([hit_rate_at_k(recommendations[u], set(ground_truth[u]), k) for u in users])
    m = np.mean([average_precision_at_k(recommendations[u], set(ground_truth[u]), k) for u in users])
    n = np.mean([ndcg_at_k(recommendations[u], set(ground_truth[u]), k) for u in users])
    return {
        f"Precision@{k}": float(p),
        f"Recall@{k}": float(r),
        f"HitRate@{k}": float(h),
        f"MAP@{k}": float(m),
        f"NDCG@{k}": float(n),
        "users_evaluated": len(users),
    }
'''
# Patch DATA_DIR to point at /content/data instead of repo-relative
DATA_PY = DATA_PY.replace('DATA_DIR = Path(__file__).resolve().parents[1] / "data"',
                          'DATA_DIR = Path("/content/data")')
with open('/content/src/__init__.py','w') as f: f.write('')
with open('/content/src/data.py','w') as f: f.write(DATA_PY)
with open('/content/src/metrics.py','w') as f: f.write(METRICS_PY)
import sys
sys.path.insert(0, '/content')
print('src/ ready in /content')


# 05 — Neural Collaborative Filtering (NeuMF)

**H&M Personalized Fashion Recommendations**

BTEC L6 Unit 2 — Capstone Project · Lola Toirxonova (ID 220062)

---

## Algorithm — NeuMF (He et al., 2017)

NeuMF replaces the inner product of matrix factorisation with a learned non-linear function. It has two parallel paths:

- **GMF (Generalised Matrix Factorisation)** — element-wise product of user and item embeddings, then a linear layer. Mathematically equivalent to MF when the weights are uniform.
- **MLP** — concatenation of user and item embeddings passed through a feed-forward network (tower with decreasing widths).

The two paths are concatenated and a final sigmoid layer outputs $P(\text{interaction} \mid u, i) \in [0, 1]$. Trained with **binary cross-entropy** on positive interactions + **uniformly sampled negatives**.

## Distinction-level critical engagement (D1 / D3)

**Rendle et al. (2020) — "Neural Collaborative Filtering vs. Matrix Factorization Revisited"** showed that a carefully tuned MF can match or beat NCF on common benchmarks. The empirical comparison in this notebook makes that finding testable on the H&M dataset. **Reporting honestly whether NCF wins here is what earns D3.**

## Hardware

- **Recommended:** Google Colab with free T4 GPU. The notebook detects the device automatically.
- **Local CPU fallback:** works but slow (10–30 minutes per epoch on a laptop). Halve the sample size if running locally.

## Criteria addressed

P4, P5, M3, M4, **D3** (NCF is the algorithm specifically called out as required for Distinction in the Student Guide).

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('/content')
sys.path.insert(0, str(REPO_ROOT))

import json
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from src import data as dataio
from src import metrics as metricslib

sns.set_theme(style='whitegrid')

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'neural_cf'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TOP_K = 10
LAST_MONTHS = 6  # last 6 months — matches H&M competition protocol  # halve to 500_000 if running on CPU

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}   |   CUDA available: {torch.cuda.is_available()}')

## 1. Load data and split (matches Notebooks 02–04)

In [ ]:
articles = dataio.load_articles()
transactions = dataio.load_transactions(last_months=LAST_MONTHS)
train, test = dataio.time_based_split(transactions, cutoff_days=7)
ground_truth = test.groupby('customer_id')['article_id'].apply(set).to_dict()
print(f'Train: {len(train):,}   |   Test: {len(test):,}')

## 2. Build integer ID encodings

Neural embeddings need contiguous integer IDs.

In [ ]:
user_index = pd.Index(train['customer_id'].unique(), name='customer_id')
item_index = pd.Index(train['article_id'].unique(), name='article_id')
user_id_to_idx = {u: i for i, u in enumerate(user_index)}
item_id_to_idx = {a: i for i, a in enumerate(item_index)}
n_users = len(user_index)
n_items = len(item_index)
print(f'#users = {n_users:,}   |   #items = {n_items:,}')

In [ ]:
train_u = train['customer_id'].map(user_id_to_idx).values.astype(np.int64)
train_i = train['article_id'].map(item_id_to_idx).values.astype(np.int64)
user_seen_items = {u: set() for u in range(n_users)}
for u, i in zip(train_u, train_i):
    user_seen_items[u].add(i)
print(f'Positive interactions: {len(train_u):,}')

## 3. Dataset with on-the-fly negative sampling

In [ ]:
class NCFDataset(Dataset):
    def __init__(self, pos_u, pos_i, n_items, user_seen_items, n_negatives=4, seed=42):
        self.pos_u = pos_u
        self.pos_i = pos_i
        self.n_items = n_items
        self.seen = user_seen_items
        self.n_negatives = n_negatives
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.pos_u) * (1 + self.n_negatives)

    def __getitem__(self, idx):
        pos_idx = idx // (1 + self.n_negatives)
        slot = idx % (1 + self.n_negatives)
        u = int(self.pos_u[pos_idx])
        if slot == 0:
            i = int(self.pos_i[pos_idx])
            label = 1.0
        else:
            # uniform random negative not in user's seen set
            while True:
                i = int(self.rng.integers(0, self.n_items))
                if i not in self.seen[u]:
                    break
            label = 0.0
        return u, i, label

## 4. NeuMF model

In [ ]:
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, gmf_dim=16, mlp_dim=32, mlp_layers=(64, 32, 16, 8), dropout=0.1):
        super().__init__()
        # GMF path
        self.user_gmf = nn.Embedding(n_users, gmf_dim)
        self.item_gmf = nn.Embedding(n_items, gmf_dim)
        # MLP path
        self.user_mlp = nn.Embedding(n_users, mlp_dim)
        self.item_mlp = nn.Embedding(n_items, mlp_dim)
        layers = []
        prev = mlp_dim * 2
        for h in mlp_layers:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.mlp = nn.Sequential(*layers)
        # Fusion + output
        self.output = nn.Linear(gmf_dim + mlp_layers[-1], 1)
        self._init_weights()

    def _init_weights(self):
        for m in [self.user_gmf, self.item_gmf, self.user_mlp, self.item_mlp]:
            nn.init.normal_(m.weight, std=0.01)
        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        nn.init.xavier_uniform_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, u, i):
        gmf = self.user_gmf(u) * self.item_gmf(i)
        mlp = self.mlp(torch.cat([self.user_mlp(u), self.item_mlp(i)], dim=-1))
        x = torch.cat([gmf, mlp], dim=-1)
        return self.output(x).squeeze(-1)

## 5. Training loop

In [ ]:
EPOCHS = 5            # raise to 10–15 if you have GPU time
BATCH_SIZE = 2048
LR = 1e-3
N_NEGATIVES = 4
WEIGHT_DECAY = 1e-6

model = NeuMF(n_users, n_items).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

dataset = NCFDataset(train_u, train_i, n_items, user_seen_items, n_negatives=N_NEGATIVES, seed=RANDOM_SEED)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

print(f'Training samples per epoch (incl. negatives): {len(dataset):,}')
print(f'Batches per epoch: {len(loader):,}')

In [ ]:
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    n_seen = 0
    for u, i, y in loader:
        u = u.to(DEVICE, non_blocking=True)
        i = i.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad()
        logits = model(u, i)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * y.size(0)
        n_seen += y.size(0)
    elapsed = time.time() - t0
    avg = running_loss / n_seen
    history.append({'epoch': epoch, 'loss': avg, 'time_sec': elapsed})
    print(f'Epoch {epoch}/{EPOCHS}   loss={avg:.4f}   time={elapsed:.1f}s')

pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
h = pd.DataFrame(history)
ax.plot(h['epoch'], h['loss'], marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.set_title('NeuMF training loss')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_loss.png', dpi=150)
plt.show()

## 6. Top-K recommendation

For each evaluated user, score all `n_items` and pick the top-K not in their train history.

In [ ]:
@torch.no_grad()
def recommend_for_user(model, user_idx, n_items, seen_set, k=10, batch=8192):
    model.eval()
    item_ids = torch.arange(n_items, device=DEVICE)
    user_ids = torch.full_like(item_ids, fill_value=user_idx)
    scores = torch.empty(n_items, device=DEVICE)
    for s in range(0, n_items, batch):
        e = min(s + batch, n_items)
        scores[s:e] = model(user_ids[s:e], item_ids[s:e])
    scores = scores.cpu().numpy()
    if seen_set:
        scores[list(seen_set)] = -np.inf
    top = np.argpartition(-scores, k)[:k]
    top = top[np.argsort(-scores[top])]
    return top.tolist()

## 7. Evaluate on the test set

In [ ]:
warm_test_users = [u for u in ground_truth if u in user_id_to_idx]
cold_test_users = [u for u in ground_truth if u not in user_id_to_idx]

EVAL_USER_CAP = 2_000
rng = np.random.default_rng(RANDOM_SEED)
if len(warm_test_users) > EVAL_USER_CAP:
    warm_eval = list(rng.choice(warm_test_users, size=EVAL_USER_CAP, replace=False))
else:
    warm_eval = warm_test_users
print(f'Evaluating on {len(warm_eval):,} warm users')

In [ ]:
idx_to_item_id = {i: a for a, i in item_id_to_idx.items()}
recommendations = {}
t0 = time.time()
for n, u in enumerate(warm_eval, 1):
    u_idx = user_id_to_idx[u]
    seen = user_seen_items[u_idx]
    top_idx = recommend_for_user(model, u_idx, n_items, seen, k=TOP_K)
    recommendations[u] = [idx_to_item_id[i] for i in top_idx]
    if n % 200 == 0:
        print(f'  {n}/{len(warm_eval)}   elapsed={time.time() - t0:.1f}s')
print(f'Done. Generated {len(recommendations):,} recommendation lists in {time.time() - t0:.1f}s')

In [ ]:
metrics_warm = metricslib.evaluate(recommendations, ground_truth, k=TOP_K)
metrics_warm

## 8. Save model artefacts

In [ ]:
torch.save(model.state_dict(), MODEL_DIR / 'ncf_neumf.pt')
with open(MODEL_DIR / 'ncf_id_maps.pkl', 'wb') as f:
    pickle.dump({
        'user_id_to_idx': user_id_to_idx,
        'item_id_to_idx': item_id_to_idx,
        'idx_to_item_id': idx_to_item_id,
        'n_users': n_users,
        'n_items': n_items,
    }, f)

results = {
    'model': 'neumf',
    'hyperparameters': {
        'gmf_dim': 16,
        'mlp_dim': 32,
        'mlp_layers': [64, 32, 16, 8],
        'dropout': 0.1,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'n_negatives': N_NEGATIVES,
        'weight_decay': WEIGHT_DECAY,
    },
    'device': str(DEVICE),
    'sample_size_transactions': len(transactions),
    'n_users': n_users,
    'n_items': n_items,
    'top_k': TOP_K,
    'eval_warm_users': len(warm_eval),
    'eval_cold_users': len(cold_test_users),
    'metrics_warm': metrics_warm,
    'training_history': history,
}
with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('NCF artefacts saved.')

## 9. Four-way comparison

Pull metrics from all four notebooks for the final head-to-head: **Content-Based → ALS CF → Hybrid → NeuMF**.

In [ ]:
def safe_load(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return None

cb = safe_load(REPO_ROOT / 'outputs' / 'content_based' / 'results.json')
als = safe_load(REPO_ROOT / 'outputs' / 'collaborative_filtering' / 'results.json')
hyb = safe_load(REPO_ROOT / 'outputs' / 'hybrid' / 'results.json')

rows = {}
if cb: rows['Content-Based'] = cb['metrics_warm']
if als: rows['ALS CF'] = als['metrics_warm']
if hyb:
    best_alpha = hyb['best_alpha']
    rows[f'Hybrid (α={best_alpha})'] = hyb['sweep'][str(best_alpha)]
rows['NeuMF'] = metrics_warm

comparison = pd.DataFrame(rows).T
metric_cols = [c for c in comparison.columns if c.startswith(('Precision@', 'Recall@', 'NDCG@', 'MAP@', 'HitRate@'))]
comparison = comparison[metric_cols].astype(float)
print('Four-way comparison (warm users):')
print(comparison.round(4))
comparison.to_csv(OUTPUT_DIR / 'four_way_comparison.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
comparison[[c for c in metric_cols if c in (f'Precision@{TOP_K}', f'Recall@{TOP_K}', f'NDCG@{TOP_K}', f'MAP@{TOP_K}')]].plot(kind='bar', ax=ax, rot=0)
ax.set_title('Four-way model comparison — warm users')
ax.set_ylabel('Score')
ax.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'four_way_comparison.png', dpi=150)
plt.show()

## 10. Findings — what to write in the report

**Three paragraphs in Chapter 5 (Findings):**

1. **Does NCF beat the classical hybrid?** State the answer plainly with the metric numbers. If yes — by how much, and at what compute cost (training time, hardware needed). If no — say so and refer to Rendle et al. (2020).
2. **Where does each model win?** It is common for NCF to win NDCG while a simpler model wins Precision@K, or vice versa. Identify the trade-offs concretely.
3. **Practical implications.** Would a small Uzbek marketplace get a meaningful uplift from deploying NCF over ALS, given the engineering cost (PyTorch in production, GPU inference)? An honest cost–benefit discussion is **M5 + D3 evidence**.

## Next steps

1. Save the comparison figure into `outputs/figures/` for use in the final report (Chapter 5).
2. Add a 3-paragraph write-up to `docs/findings_ncf.md`.
3. Update `logbook.md` with the sprint progress.
4. Move on to `notebooks/06_evaluation.ipynb` — 5-fold cross-validation with statistical significance tests.